In [1]:
import os, json
import requests
import numpy as np

stats_dir = 'stats'
os.makedirs(stats_dir, exist_ok=True)

multimericities = np.arange(2, 7)

seq_len_limit = 600


In [ ]:
# ### GET ASSEMBLIES FOR HOMOMERIC MULTIMERS

# tallies = []

# # READ TEMPLATE SEARCH API PAYLOAD
# search_api_payload_file = 'template - search_api - homo-multimers.json'
# with open(search_api_payload_file, 'r') as f:
#     search_api_template = f.read()

# # GET ASSEMBLIES FOR HOMOMERIC MULTIMERS
# for multimericity in multimericities:
#     print(f'{multimericity}-mer', end='')

#     # prepare url
#     payload = search_api_template.replace(
#         'MULTIMERIC_PLACEHOLDER', str(multimericity)
#     )
#     payload = payload.replace('RESOLUTION_PLACEHOLDER', '2')
#     url = f'https://search.rcsb.org/rcsbsearch/v2/query?json={payload}'

#     # GET request
#     response = requests.get(url)

#     # decode returned response if request is successful
#     if response.status_code != 200:
#         print(response.text)
#     decoded = response.json()
#     n_entries_retrieved = decoded['total_count']
#     tallies.append(n_entries_retrieved)

#     print(f' -> {n_entries_retrieved} assemblies received from RCSB')

#     # convert decoded response into lists
#     assemblies = [entry['identifier'] for entry in decoded['result_set']]

#     # save list to corresponding file
#     np.savetxt(
#         f'{stats_dir}/OUT-1-1.assemblies-homo_{multimericity}mer.txt',
#         assemblies, fmt='%s'
#     )

# print()
# print(tallies, sum(tallies))

2-mer -> 31889 assemblies received from RCSB
3-mer -> 3703 assemblies received from RCSB
4-mer -> 7154 assemblies received from RCSB
5-mer -> 373 assemblies received from RCSB
6-mer -> 1700 assemblies received from RCSB

[31889, 3703, 7154, 373, 1700] 44819


In [ ]:
# ### DOWNLOAD SEQUENCES OF ASSEMBLIES (TO DO SEQUENCE SIMILARITY TRIAGE)

# # READ TEMPLATE SEARCH API PAYLOAD
# data_api_payload_file = 'template - data_api - sequences.txt'
# with open(data_api_payload_file, 'r') as f:
#     data_api_template = f.read()

# for multimericity in multimericities:
#     print(f'{multimericity}-mer', end='')

#     assemblies = np.loadtxt(
#         f'{stats_dir}/OUT-1-1.assemblies-homo_{multimericity}mer.txt',
#         dtype=np.str_
#     )

#     # REDUCE NUMBER OF ENTRIES IN QUERY TO SATISFY API REQUIREMENTS
#     all_decoded = {'data': {'assemblies': []}}
#     for split_idx in range(len(assemblies)//10000+1):
#         idx_start = split_idx * 10000
#         idx_end   = (split_idx+1) * 10000

#         # GET request
#         payload = data_api_template.replace(
#             'ASSEMBLY_LIST_PLACEHOLDER',
#             '["' + '","'.join(assemblies[idx_start:idx_end]) + '"]'
#         )
#         url = f'https://data.rcsb.org/graphql'
#         response = requests.post(url, json={'query': payload})

#         # decode returned response if request is successful
#         if response.status_code != 200:
#             print(response.text)
#         decoded = response.json()

#         # merge all data for multimericity
#         all_decoded['data']['assemblies'] += decoded['data']['assemblies']

#     # SAVE DATA TO CORRESPONDING FILE
#     file = f'{stats_dir}/OUT-1-2.assembly_data-homo_{multimericity}mer.json'
#     with open(file, 'w+', encoding='utf-8') as f:
#         json.dump(all_decoded, f, indent=2, sort_keys=True)

#     print(f' -> Done ({len(all_decoded["data"]["assemblies"])} entries saved)')


2-mer -> Done (31889 entries saved)
3-mer -> Done (3703 entries saved)
4-mer -> Done (7154 entries saved)
5-mer -> Done (373 entries saved)
6-mer -> Done (1700 entries saved)


In [2]:
### GATHER ALL INFORMATION BY ENTRY (MERGING ALL MULTIMERICITIES)

header = [
    'full_id',
    'pdb_id',
    'assembly_id',
    'multimericity',
    'auth_chain_id',
    'asym_chain_id',
    'seq_database',
    'seq_database_accession',
    'sequence_in_pdb'
]
all_info = []

# record entries to be discarded later
entries_with_more_than_one_assembly = {
    multimericity: [] for multimericity in multimericities
}
entries_not_on_uniprot = []
entries_with_chain_number_mismatch = []
entries_too_long = []

for multimericity in multimericities:
    print(f'{multimericity}-mers', end='')

    # READ DATA
    file = f'{stats_dir}/OUT-1-2.assembly_data-homo_{multimericity}mer.json'
    with open(file, 'r') as f:
        data = json.load(f)
    assemblies = data['data']['assemblies']

    # EXTRACT INFO ASSEMBLY BY ASSEMBLY
    for assembly_idx, assembly in enumerate(assemblies):
        full_id = assembly['rcsb_id']
        pdb_id, assembly_id = full_id.split('-')

        # CHECK IF ENTRY INFO INCLUDES MORE THAN ONE ASSEMBLY
        if len(assembly['entry']['polymer_entities']) != 1:
            entries_with_more_than_one_assembly[multimericity].append(full_id)

        # GATHER INFO FOR ENTRY
        entry_info = assembly['entry']['polymer_entities'][0]
        id_info = entry_info['rcsb_polymer_entity_container_identifiers']

        sequence_in_pdb = entry_info['entity_poly']['pdbx_seq_one_letter_code_can']
        if len(sequence_in_pdb) > seq_len_limit:
            entries_too_long.append(full_id)

        asym_chain_id = id_info['asym_ids']
        auth_chain_id = id_info['auth_asym_ids']
        if len(asym_chain_id) != len(auth_chain_id):
            entries_with_chain_number_mismatch.append(full_id)

        if id_info['reference_sequence_identifiers'] is not None:
            seq_database = id_info['reference_sequence_identifiers'][0]['database_name']
            seq_database_accession = id_info['reference_sequence_identifiers'][0]['database_accession']
        else:
            seq_database = 'none'
            seq_database_accession = 'none'
        if seq_database != 'UniProt':
            entries_not_on_uniprot.append(full_id)

        all_info.append([
            full_id,
            pdb_id,
            assembly_id,
            multimericity,
            '_'.join(auth_chain_id),
            '_'.join(asym_chain_id),
            seq_database,
            seq_database_accession,
            sequence_in_pdb
        ])

    print(' -> Done')

all_info = np.array(all_info)

print()
print('# entries in total:', len(all_info))
_, cnt = np.unique(
    all_info[:, header.index('multimericity')],
    return_counts=True
)
print(cnt, np.sum(cnt))


2-mers -> Done
3-mers -> Done
4-mers -> Done
5-mers -> Done
6-mers -> Done

# entries in total: 44819
[31889  3703  7154   373  1700] 44819


In [ ]:
### PRELIMINARY FILTERS
print('ENTRIES DISCARDED')

all_ids = all_info[:, header.index('full_id')]

# REMOVE ANY ENTRIES THAT ARE FOUND WITH MORE THAN ONE MULTIMERICITY
all_ids_unique, count = np.unique(all_ids, return_counts=True)
entries_with_multiple_multimericities = all_ids_unique[count > 1]

# SUMMARIZE
print()
print(f'entries with multiple multimericities: {len(entries_with_multiple_multimericities)}')

print()
total = 0
for multimericity in multimericities:
    print(f'{multimericity}-mers with over one assembly: {len(entries_with_more_than_one_assembly[multimericity])}')
    print(entries_with_more_than_one_assembly[multimericity][:5])
    total += len(entries_with_more_than_one_assembly[multimericity])
print(f'Total: {total}')

print()
print(f'Chain number mismatch: {len(entries_with_chain_number_mismatch)}')

print()
print(f'Sequence database not UniProt: {len(entries_not_on_uniprot)}')

print()
print(f'Sequence too long: {len(entries_too_long)}')

print()
entries_to_discard = np.unique(
    [e for v in entries_with_more_than_one_assembly.values() for e in v]
    + entries_not_on_uniprot
    + entries_with_chain_number_mismatch
    + entries_with_multiple_multimericities.tolist()
    + entries_too_long
)
print('Total:', len(entries_to_discard))

all_info = all_info[~np.isin(all_ids, entries_to_discard)]
print()
_, cnt = np.unique(
    all_info[:, header.index('multimericity')].astype(int),
    return_counts=True
)
print('# entries after discarding')
print(cnt, np.sum(cnt))

ENTRIES DISCARDED

entries with multiple multimericities: 0

2-mers with over one assembly: 2148
['12E8-1', '1A6V-1', '1A6V-3', '1ABW-1', '1ALX-1']
3-mers with over one assembly: 134
['1BB1-1', '1CZQ-1', '1CZY-1', '1D01-1', '1D0A-3']
4-mers with over one assembly: 539
['1A00-1', '1A01-1', '1A0Z-1', '1A3N-1', '1A3O-1']
5-mers with over one assembly: 26
['1LT3-1', '1LT4-1', '1LTS-1', '1R4P-1', '1S5D-1']
6-mers with over one assembly: 140
['1AIK-1', '1BEN-13', '1CPC-4', '1CPC-5', '1JEK-1']
Total: 2987

Chain number mismatch: 0

Sequence database not UniProt: 1669

Sequence too long: 1518

Total: 5455

# entries after discarding
[27856  3228  6151   317  1421] 38973


In [6]:
### SAVE

np.savetxt(
    f'{stats_dir}/OUT-1-3.all_information.csv',
    all_info[:,:-1],
    delimiter=',',
    fmt='%s',
    header=','.join(header[:-1]),
)

np.savetxt(
    f'{stats_dir}/OUT-1-3.entries_discarded.txt',
    entries_to_discard,
    fmt='%s',
)

# fasta
all_ids = all_info[:, header.index('full_id')]
all_sequences = all_info[:, header.index('sequence_in_pdb')]
all_auth_chain_id = all_info[:, header.index('auth_chain_id')]
all_asym_chain_id = all_info[:, header.index('asym_chain_id')]
with open(f'{stats_dir}/OUT-1-3.all_sequences.fasta', 'w') as f:
    for assembly_idx in range(len(all_sequences)):
        f.write(
            f'>{all_ids[assembly_idx]},'
            f'auth_chain_id:{all_auth_chain_id[assembly_idx]},'
            f'asym_chain_id:{all_asym_chain_id[assembly_idx]}\n'
        )
        f.write(f'{all_sequences[assembly_idx]}\n')
